In [6]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [32]:
%pip install pandasql

  Using cached pandasql-0.7.3-py3-none-any.whl
  Using cached sqlalchemy-2.1.5-cp314-cp314-win_amd64.whl.metadata (9.9 kB)
Using cached sqlalchemy-2.1.5-cp314-cp314-win_amd64.whl (2.4 MB)

   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 2/2 [pandasql]

Note: you may need to restart the kernel to use updated packages.


In [37]:
import streamlit as st
import pandas as pd
import pandasql as ps

# function query(sql) will help us write our SQL queries easily
def query(sql):
    return ps.sqldf(sql, globals())

#st.write("mrazq gi2")


In [12]:
dataset_path = "data-6984edcd-f9bf-4416-b47e-a61a0c370d52.csv" 

dataset = pd.read_csv(dataset_path)

In [14]:
print("Number of rows and columns:", dataset.shape)
print("\nColumn names:")
for col in dataset.columns:
    print(" -", col)
print("\nPreview of the data:")
dataset.head()
#Inspecting the dataset columns and rows to check for column types, missing values, outliers, inconsistencies

Number of rows and columns: (400, 16)

Column names:
 - salary_min
 - salary_max
 - required_experience_years
 - skills_listed
 - vague_word_count
 - exclamation_marks
 - allcaps_ratio
 - posting_hour
 - employer_response_delay_h
 - is_remote
 - free_mail_contact
 - company_age_months
 - company_size
 - category
 - mentions_trial_or_payment
 - scam

Preview of the data:


,salary_min,salary_max,required_experience_years,skills_listed,vague_word_count,exclamation_marks,allcaps_ratio,posting_hour,employer_response_delay_h,is_remote,free_mail_contact,company_age_months,company_size,category,mentions_trial_or_payment,scam
0,NaN,NaN,6.7,17,3,0,0.033,21,9.5,0,0,29.0,4,admin,0,0
1,NaN,NaN,8.8,26,3,2,0.013,8,67.8,1,1,70.0,1,healthcare,1,0
2,4571.0,12829.0,2.1,7,1,0,0.121,3,4.6,0,1,29.0,3,hospitality,0,1
3,NaN,NaN,8.1,20,3,1,0.028,7,70.2,1,0,147.0,8,logistics,0,0
4,5807.0,7104.0,1.8,23,3,1,0.021,10,11.6,0,1,7.0,7,hospitality,0,0


In [ ]:
query("""
SELECT is_remote,
       AVG(scam) AS avg_scamvalue
FROM dataset
GROUP BY is_remote;
""")

,is_remote,avg_scamvalue
0,0,0.245536
1,1,0.306818


In [39]:
query("""
SELECT exclamation_marks,
       AVG(scam) AS avg_scamvalue
FROM dataset
GROUP BY exclamation_marks;
""")

,exclamation_marks,avg_scamvalue
0,0,0.263158
1,1,0.196970
2,2,0.358491
3,3,0.333333
4,4,0.285714
5,5,0.000000


In [40]:
query("""
SELECT skills_listed,
       AVG(scam) AS avg_scamvalue
FROM dataset
GROUP BY skills_listed;
""")

,skills_listed,avg_scamvalue
0,4,1.000000
1,5,1.000000
2,6,1.000000
3,7,1.000000
4,8,1.000000
5,9,0.521739
6,10,0.363636
7,11,0.314286
8,12,0.423077
9,13,0.000000


1st 2 summary queries do not correlate as much, whereas the 3rd one has a positive correlation - the more skills, the less likely it is a scam. Thus we can conclude that the 1st 2 columns can be cleaned up from the features for our prediction model.

In [ ]:
dataset_encoded=pd.get_dummies(dataset)
dataset_encoded = dataset_encoded.astype(float)
y=dataset_encoded["scam"]

X=dataset_encoded.drop(dataset["scam"])
print(y,'\n')
print(X)
#Checking if encoding worked

0      0.0
1      0.0
2      1.0
3      0.0
4      0.0
      ... 
395    0.0
396    0.0
397    1.0
398    0.0
399    1.0
Name: scam, Length: 400, dtype: float64 

     salary_min  salary_max  required_experience_years  skills_listed  \
2        4571.0     12829.0                        2.1            7.0   
3           NaN         NaN                        8.1           20.0   
4        5807.0      7104.0                        1.8           23.0   
5        4872.0      6485.0                        4.4           11.0   
6           NaN         NaN                        6.2           11.0   
..          ...         ...                        ...            ...   
395      6257.0      8286.0                        9.5           16.0   
396      5720.0      7641.0                        1.9           20.0   
397      3252.0      5420.0                        3.8            9.0   
398         NaN         NaN                        5.2           18.0   
399      5805.0     12723.0       

In [ ]:
print(dataset_encoded.columns.tolist())
# Checking column names (for catergories)

['salary_min', 'salary_max', 'required_experience_years', 'skills_listed', 'vague_word_count', 'exclamation_marks', 'allcaps_ratio', 'posting_hour', 'employer_response_delay_h', 'is_remote', 'free_mail_contact', 'company_age_months', 'company_size', 'mentions_trial_or_payment', 'scam', 'category_admin', 'category_finance', 'category_healthcare', 'category_hospitality', 'category_it', 'category_logistics', 'category_marketing', 'category_sales']


In [20]:
correlations = dataset_encoded.corr()["scam"].drop("scam")

correlations_ranked = correlations.abs().sort_values(ascending=False)

print(correlations.loc[correlations_ranked.index])

allcaps_ratio                0.724263
skills_listed               -0.674678
required_experience_years   -0.624439
posting_hour                -0.466398
salary_max                   0.393568
company_size                -0.341584
salary_min                  -0.331889
mentions_trial_or_payment    0.219613
company_age_months          -0.209320
category_finance             0.195160
vague_word_count             0.172289
employer_response_delay_h   -0.170519
category_sales              -0.135917
category_it                  0.096186
category_healthcare         -0.079117
exclamation_marks            0.070987
is_remote                    0.068321
category_logistics          -0.048953
free_mail_contact            0.035808
category_hospitality         0.035003
category_admin              -0.031517
category_marketing          -0.019534
Name: scam, dtype: float64


Every column with a correlation under +/-0.2 is practically useless for the regressor so they will be cleaned and removed from the dataset.

In [ ]:
features_to_remove = ["category_finance", "vague_word_count",
"employer_response_delay_h", "category_sales", "category_it",
"category_healthcare", "exclamation_marks",
"is_remote", "category_logistics",
"free_mail_contact", "category_hospitality",
"category_admin", "category_marketing"]

for feature in features_to_remove:
 dataset_encoded=dataset_encoded.drop(feature, axis=1)
 
# Cleaning step (removing all unnecessary columns that would be harmful/useless for the prediction model) based on correlation table

In [ ]:
print(dataset_encoded.columns.tolist())
# Checking if cleaning step was executed properly

['salary_min', 'salary_max', 'required_experience_years', 'skills_listed', 'allcaps_ratio', 'posting_hour', 'company_age_months', 'company_size', 'mentions_trial_or_payment', 'scam']


In [ ]:
query("""
SELECT * FROM dataset_encoded""")
#In summary - the query shows it works and all unneeded columns are cleaned

,salary_min,salary_max,required_experience_years,skills_listed,allcaps_ratio,posting_hour,company_age_months,company_size,mentions_trial_or_payment,scam
0,NaN,NaN,6.7,17.0,0.033,21.0,29.0,4.0,0.0,0.0
1,NaN,NaN,8.8,26.0,0.013,8.0,70.0,1.0,1.0,0.0
2,4571.0,12829.0,2.1,7.0,0.121,3.0,29.0,3.0,0.0,1.0
3,NaN,NaN,8.1,20.0,0.028,7.0,147.0,8.0,0.0,0.0
4,5807.0,7104.0,1.8,23.0,0.021,10.0,7.0,7.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
395,6257.0,8286.0,9.5,16.0,0.017,10.0,20.0,13.0,0.0,0.0
396,5720.0,7641.0,1.9,20.0,0.010,12.0,11.0,3.0,1.0,0.0
397,3252.0,5420.0,3.8,9.0,0.127,1.0,63.0,1.0,0.0,1.0
398,NaN,NaN,5.2,18.0,0.021,10.0,96.0,42.0,0.0,0.0


In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_s_last, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)

plt.title("KNN Regression: Actual vs Predicted Exam Scores")
plt.xlabel("Actual Exam Score")
plt.ylabel("Predicted Exam Score")
plt.grid(True)
plt.show()